# SOLikeT in a few lines

SOLikeT provides cosmology likelihoods for the [Cobaya](https://cobaya.readthedocs.io) sampler.
This notebook is the **30-second on-ramp**: it gets you a working likelihood you can poke at,
using the `soliket.presets` helpers.

> For the full, hand-built configuration (parameter blocks, theory wiring, YAML, running MCMC)
> see [`first_step_tutorial.ipynb`](first_step_tutorial.ipynb) — the detailed deep-dive.

> **First run:** the MFLike likelihood needs its data (~500 MB, fetched from NERSC), so the cell
> below installs it before anything else uses it. It is a no-op once the data is present, so it is
> safe to re-run. `_nb` is `notebooks/_nb.py`, the repo-only notebook helpers.

In [ ]:
import _nb

_nb.install_data("mflike")  # downloads the data on first run; a no-op afterwards

## 1. A working likelihood, instantly

`quickstart` wires a preset's likelihood + theory + fiducial parameters into a Cobaya model and
returns a `Session`. Available presets: `mflike`, `lensing`, `multigaussian`.

In [ ]:
from soliket.presets import quickstart

s = quickstart("mflike")  # MFLike TT/TE/EE + foreground + CAMB, at the fiducial point
print(s.loglike())  # evaluate the log-likelihood

## 2. Reach the pieces by name

No fragile `model.components[3]` indexing — the `Session` exposes role aliases.

In [ ]:
print("likelihood :", type(s.mflike).__name__)
print("foreground :", type(s.foreground).__name__)
print("cosmology  :", type(s.cosmo).__name__)

## 3. Plot the theory spectrum

`notebooks/_nb.py` holds notebook-only conveniences (plotting, data download).

In [ ]:
import _nb

dls = _nb.theory_dls(s)  # CMB D_ell from the evaluated model
_nb.plot_dls(dls, spectra=("tt", "te", "ee"));

## 4. Vary a parameter

Parameters are *fixed at fiducial* by default. Pass `sample=[...]` to turn a parameter into a
sampled one (it gets its prior back), then run an MCMC with `s.run()` (Cobaya, Python-native).

In [ ]:
s_tau = quickstart("mflike", sample=["tau"])
# s_tau.run()
print("tau is now sampled:", "prior" in s_tau.info["params"]["tau"])

In [ ]:
import numpy as np
from tqdm import tqdm

tau_start = 0.053
tau_end = 0.057
tau_steps = 20
tau_values = np.linspace(tau_start, tau_end, tau_steps)

chi2_values = np.empty_like(tau_values)
for i in tqdm(range(tau_steps), desc="Evaluating likelihood for different tau values"):
    chi2_values[i] = -2 * s_tau.loglike(point={"tau": tau_values[i]})

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].plot(tau_values, chi2_values - np.min(chi2_values))
ax[0].set_xlabel(r"$\tau$")
ax[0].set_ylabel(r"$\chi^2$")

ax[1].plot(tau_values, np.exp(-0.5 * (chi2_values - np.min(chi2_values))))
ax[1].set_xlabel(r"$\tau$")
ax[1].set_ylabel("Relative probability")

plt.show()

## 5. Evaluate the likelihood on a 2D grid of parameter values, and plot the results.

In [ ]:
s_logA_H0 = quickstart("mflike", sample=["logA", "H0"])
print(
    "logA and H0 are now sampled:",
    "prior" in s_logA_H0.info["params"]["logA"],
    "prior" in s_logA_H0.info["params"]["H0"],
)

In [ ]:
logA_H0_start = [3.0435, 67.225]
logA_H0_end = [3.048, 67.425]
logA_H0_steps = 10

logA_values = np.linspace(logA_H0_start[0], logA_H0_end[0], logA_H0_steps)
H0_values = np.linspace(logA_H0_start[1], logA_H0_end[1], logA_H0_steps)
logA_grid, H0_grid = np.meshgrid(logA_values, H0_values)
chi2_grid = np.empty_like(logA_grid)

for i in tqdm(
    range(logA_H0_steps), desc="Evaluating likelihood for different logA and H0 values"
):
    for j in range(logA_H0_steps):
        chi2_grid[i, j] = -2 * s_logA_H0.loglike(
            point={"logA": logA_grid[i, j], "H0": H0_grid[i, j]}
        )

In [ ]:
plt.imshow(
    np.exp(-0.5 * (chi2_grid - np.min(chi2_grid))),
    extent=[logA_H0_start[0], logA_H0_end[0], logA_H0_start[1], logA_H0_end[1]],
    origin="lower",
    aspect="auto",
)
plt.colorbar(label="Relative probability")
plt.xlabel(r"$\log A$")
plt.ylabel(r"$H_0$")

plt.show()

In [ ]:
from scipy import interpolate

oversample_factor = 10
logA_grid_fine = np.linspace(
    logA_H0_start[0], logA_H0_end[0], logA_H0_steps * oversample_factor
)
H0_grid_fine = np.linspace(
    logA_H0_start[1], logA_H0_end[1], logA_H0_steps * oversample_factor
)

splined_chi2 = interpolate.RectBivariateSpline(
    logA_values, H0_values, chi2_grid.T, kx=3, ky=3, s=0
)
chi2_grid_fine = splined_chi2(logA_grid_fine, H0_grid_fine).T

In [ ]:
normalized_chi2 = np.exp(-0.5 * (chi2_grid_fine - np.min(chi2_grid_fine)))

def density_levels(pdf, fracs=(0.68, 0.95)):
    flat = np.sort(pdf.ravel())[::-1]
    cum = np.cumsum(flat)
    cum /= cum[-1]
    return [flat[np.searchsorted(cum, f)] for f in fracs]

levels = sorted(density_levels(normalized_chi2))
plt.contour(logA_grid_fine, H0_grid_fine, normalized_chi2, levels=levels, colors="black")

plt.contour(
    logA_grid_fine,
    H0_grid_fine,
    normalized_chi2,
    levels=levels,
    colors="black",
    linewidths=0.8,
)

plt.imshow(
    normalized_chi2,
    extent=[logA_H0_start[0], logA_H0_end[0], logA_H0_start[1], logA_H0_end[1]],
    origin="lower",
    aspect="auto",
)
plt.colorbar(label="Relative probability")
plt.xlabel(r"$\log A$")
plt.ylabel(r"$H_0$")

plt.show()

In [ ]:
filled_levels = levels + [normalized_chi2.max()]

plt.contourf(
    logA_grid_fine,
    H0_grid_fine,
    normalized_chi2,
    levels=filled_levels,
    colors=["#c6dbef", "#2171b5"],
    alpha=0.5,
)

plt.contour(
    logA_grid_fine,
    H0_grid_fine,
    normalized_chi2,
    levels=levels,
    colors="black",
    linewidths=0.8,
)

plt.xlabel(r"$\log A$")
plt.ylabel(r"$H_0$")

plt.show()

## 6. Run a full MCMC

In [ ]:
# uncomment to launch a full MCMC

# s_tau.run(output="chains/mflike")

## Where to go next

- **[`first_step_tutorial.ipynb`](first_step_tutorial.ipynb)** — build the same configuration by hand
  to understand every block.
- **`soliket.presets`** — `load_params`, `build_info`, `quickstart`, `resolve_aliases`.
- **`ISO_sims/`** — create simulated datasets, build cross-covariances, and run analyses.